In [1]:
"""
Method 4: Movement Pruning — ResNet-50 / CIFAR-10
==================================================
TRUE movement pruning (Sanh et al., 2020).

Core idea:
  Score(w_i) = sign(w_i(t0)) × (w_i(t_final) − w_i(t0))
  Weights that move TOWARD zero are pruned.
  Weights that move AWAY from zero are kept, even if currently small.

Requires brief fine-tuning so weights can reveal their movement direction.

Saves exactly 8 metrics per sparsity level for comparison with baseline:
  accuracy, precision, recall, f1, num_parameters, model_size_mb,
  inference_time_ms, flops

Output: method4_movement_pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "method4_movement_pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY_LEVELS   = [0.30, 0.50, 0.70, 0.80, 0.90]
FINETUNE_EPOCHS   = 3
FINETUNE_LR       = 1e-4
FINETUNE_MOMENTUM = 0.9
FINETUNE_WD       = 1e-4
INFERENCE_RUNS    = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


# ── Data ──────────────────────────────────────────────────────────────────────
def get_loaders():
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10(root="../data", train=True,
                                             download=True, transform=transform_train)
    test_ds  = torchvision.datasets.CIFAR10(root="../data", train=False,
                                             download=True, transform=transform_test)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, test_loader


# ── Fine-tuning ───────────────────────────────────────────────────────────────
def finetune(model, train_loader, n_epochs, lr):
    """
    Brief fine-tuning at a low LR so weights reveal their movement direction
    without drifting far from the pre-trained checkpoint.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=FINETUNE_MOMENTUM, weight_decay=FINETUNE_WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    model.train()
    for epoch in range(n_epochs):
        total_loss, correct, total = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss    = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += outputs.argmax(1).eq(targets).sum().item()
            total      += targets.size(0)
        scheduler.step()
        print(f"      FT epoch {epoch+1}/{n_epochs} | "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Acc: {correct/total:.4f}")
    return model


# ── Movement scoring & pruning ────────────────────────────────────────────────
def apply_movement_pruning(model, train_loader, sparsity):
    """
    1. Snapshot weights before fine-tuning  (w_before)
    2. Fine-tune briefly at low LR
    3. Snapshot weights after              (w_after)
    4. Score = sign(w_before) × (w_after − w_before)
       Positive → moved away from zero → KEEP
       Negative → moved toward zero    → PRUNE
    5. Prune lowest-scoring weights at target sparsity
    """
    pruned = copy.deepcopy(model)

    # Step 1 — snapshot
    w_before = {name: module.weight.data.clone().cpu()
                for name, module in pruned.named_modules()
                if isinstance(module, (nn.Conv2d, nn.Linear))}

    # Step 2 — fine-tune
    print(f"    Fine-tuning {FINETUNE_EPOCHS} epoch(s) at LR={FINETUNE_LR}...")
    pruned = finetune(pruned, train_loader, FINETUNE_EPOCHS, FINETUNE_LR)

    # Step 3 — snapshot after
    w_after = {name: module.weight.data.clone().cpu()
               for name, module in pruned.named_modules()
               if isinstance(module, (nn.Conv2d, nn.Linear))}

    # Step 4 — movement scores
    scores = {name: torch.sign(w_before[name]) * (w_after[name] - w_before[name])
              for name in w_before}

    # Step 5 — global threshold & mask
    all_scores = torch.cat([s.view(-1) for s in scores.values()])
    k          = max(1, int(all_scores.numel() * sparsity))
    threshold  = torch.kthvalue(all_scores, k).values.item()

    with torch.no_grad():
        for name, module in pruned.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)) and name in scores:
                mask = (scores[name].to(DEVICE) > threshold).float()
                module.weight.data *= mask

    pruned.eval()
    return pruned


# ── 8 Core Metrics ────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / (1024 ** 2)
    finally:
        os.remove(tmp)
    return size

def measure_inference_time_ms(model):
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32).to(DEVICE)

    if DEVICE.type == "cuda":
        for _ in range(20):
            model(dummy)
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / INFERENCE_RUNS
    else:
        with torch.no_grad():
            for _ in range(5):
                model(dummy)
            t0 = time.perf_counter()
            for _ in range(INFERENCE_RUNS):
                model(dummy)
        return (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

def compute_flops(model):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return int(macs * 2)


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Method 4: Movement Pruning — ResNet-50 / CIFAR-10")
    print("  Score = sign(w₀) × Δw  (Sanh et al., 2020)")
    print(f"  Device : {DEVICE}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model = load_baseline(BASELINE_MODEL_PATH)
    train_loader, test_loader = get_loaders()

    results = []

    for sparsity in SPARSITY_LEVELS:
        print(f"\n  Target sparsity: {int(sparsity*100)}%")
        pruned = apply_movement_pruning(model, train_loader, sparsity)

        metrics       = evaluate(pruned, test_loader)
        num_params    = count_parameters(pruned)
        model_size_mb = get_model_size_mb(pruned)
        inference_ms  = measure_inference_time_ms(pruned)
        flops         = compute_flops(pruned)

        row = {
            "sparsity": sparsity,
            # ── 8 standardized metrics ──
            "accuracy"         : round(metrics["accuracy"],  6),
            "precision"        : round(metrics["precision"], 6),
            "recall"           : round(metrics["recall"],    6),
            "f1"               : round(metrics["f1"],        6),
            "num_parameters"   : num_params,
            "model_size_mb"    : round(model_size_mb, 4),
            "inference_time_ms": round(inference_ms, 4),
            "flops"            : flops,
        }
        results.append(row)

        print(f"    Acc: {metrics['accuracy']:.4f} | Params: {num_params:,} | "
              f"Size: {model_size_mb:.2f} MB | Inf: {inference_ms:.2f} ms | "
              f"FLOPs: {flops/1e9:.3f} G")

    report = {
        "method"     : "movement_pruning",
        "description": ("True movement pruning (Sanh et al. 2020). "
                        "Score = sign(w_before) × (w_after − w_before). "
                        "Requires fine-tuning to observe weight movement direction."),
        "finetune_config": {
            "epochs"      : FINETUNE_EPOCHS,
            "lr"          : FINETUNE_LR,
            "momentum"    : FINETUNE_MOMENTUM,
            "weight_decay": FINETUNE_WD,
        },
        "baseline": baseline,
        "results" : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")


if __name__ == "__main__":
    main()


  Method 4: Movement Pruning — ResNet-50 / CIFAR-10
  Score = sign(w₀) × Δw  (Sanh et al., 2020)
  Device : cuda


  Target sparsity: 30%
    Fine-tuning 3 epoch(s) at LR=0.0001...
      FT epoch 1/3 | Loss: 0.1113 | Acc: 0.9882
      FT epoch 2/3 | Loss: 0.0944 | Acc: 0.9891
      FT epoch 3/3 | Loss: 0.0898 | Acc: 0.9886
    Acc: 0.1000 | Params: 23,520,842 | Size: 90.03 MB | Inf: 51.02 ms | FLOPs: 2.623 G

  Target sparsity: 50%
    Fine-tuning 3 epoch(s) at LR=0.0001...
      FT epoch 1/3 | Loss: 0.1098 | Acc: 0.9887
      FT epoch 2/3 | Loss: 0.0943 | Acc: 0.9888
      FT epoch 3/3 | Loss: 0.0884 | Acc: 0.9893
    Acc: 0.1000 | Params: 23,520,842 | Size: 90.03 MB | Inf: 51.02 ms | FLOPs: 2.623 G

  Target sparsity: 70%
    Fine-tuning 3 epoch(s) at LR=0.0001...
      FT epoch 1/3 | Loss: 0.1111 | Acc: 0.9886
      FT epoch 2/3 | Loss: 0.0931 | Acc: 0.9892
      FT epoch 3/3 | Loss: 0.0905 | Acc: 0.9884
    Acc: 0.1000 | Params: 23,520,842 | Size: 90.03 MB | Inf: 50.94 ms | FLOPs: